# 📗 LazyGraphRAG·HippoRAG와 네 방식 비교

교안 01·02는 Microsoft GraphRAG와 LightRAG를 직접 실행했습니다. 이 부록은 공개 구현이 없거나 연구 단계인 LazyGraphRAG와 HippoRAG를 개념과 측정 결과로 익히고, 네 방식의 비용과 근거를 비교합니다.

**학습 목표**

1. LazyGraphRAG가 색인 때 LLM 없이 개념 그래프를 만들고, 질문 때 예산만큼 LLM으로 관련성을 검사하는 과정을 설명합니다.
2. HippoRAG 2가 질문과 관련된 구에서 PPR로 점수를 퍼뜨려 문서를 고르는 과정과, 표기 통일이 필요한 이유를 설명합니다.
3. 네 방식의 색인 비용, 질문 1개당 비용, 질문 유형별 근거 문서 포함률을 비교합니다.

## 1. LazyGraphRAG: 색인은 LLM 없이, 질문 때 예산만큼

LazyGraphRAG(Microsoft Research, 2024)는 커뮤니티 보고서를 색인 때 미리 쓰지 않고, 질문이 오면 필요한 문서만 LLM으로 읽습니다. Lazy는 LLM 작업을 질문 때까지 미룬다는 뜻입니다.

### 1-1. 색인: 명사 개념, 공출현 그래프, 커뮤니티

색인은 LLM 없이 세 단계로 합니다.

1. **개념 뽑기:** 형태소 분석기로 문서의 명사를 뽑습니다.
2. **공출현 그래프:** 같은 청크에 함께 나온 개념끼리 잇습니다.
3. **커뮤니티:** 교안 01의 GraphRAG와 같은 Leiden 알고리즘으로 서로 많이 연결된 개념을 묶습니다.
4. **청크 배정:** 청크마다 그 청크의 개념이 가장 많이 속한 커뮤니티를 그 청크의 커뮤니티로 정합니다. 질문과 비교할 청크 임베딩도 만들어 둡니다(LLM이 아니라 임베딩 모델 호출).

<img src="images/03_lazy_index.png" width="1050" alt="LazyGraphRAG의 색인: 공지 문서에서 형태소 분석으로 명사 개념을 뽑고, 함께 나온 개념끼리 연결한 뒤, 커뮤니티로 묶습니다. 색인의 LLM 호출은 0회이고, LLM은 질문이 올 때 예산만큼만 부릅니다.">

예를 들어 안내문 「화엄사」에 `화엄사`, `지리산`, `노고단`이 함께 나오면 세 개념이 서로 이어집니다. 다른 안내문 「노고단」에도 `노고단`, `지리산`이 나오면 두 안내문의 개념이 많이 겹쳐 같은 커뮤니티로 묶이고, 두 청크도 그 커뮤니티에 배정됩니다. 32편 색인의 LLM 호출은 0회이고, 문서를 추가해도 개념 그래프만 다시 계산합니다.

### 1-2. 질문: 하위 질의, 커뮤니티 순위, 예산만큼 관련성 검사

**예산**은 질문 하나에서 LLM으로 관련성을 검사할 청크 수의 상한입니다.

1. LLM이 질문을 하위 질의 몇 개로 나눕니다.
2. 하위 질의를 임베딩해 청크 임베딩과 비교하고, 청크를 유사도 순으로 줄 세웁니다. 커뮤니티 순위는 그 커뮤니티에 속한 청크 가운데 가장 높은 순위로 정합니다.
3. 순위가 높은 커뮤니티의 청크부터 LLM에 "이 질문의 근거인가"를 묻습니다. 예산에 닿거나, 이어서 검사한 커뮤니티에서 관련 청크가 더 나오지 않으면 멈춥니다.
4. 관련 있다고 판정된 청크만 넘겨 답합니다.

**명사 개념은 질문과 맞춰 보는 데 쓰지 않습니다.** 질문과 비교하는 것은 청크 임베딩입니다. 개념 그래프는 어느 청크들이 같은 주제 묶음인지를 정해, 커뮤니티 순서대로 검사하고 관련 청크가 끊긴 묶음에서 다른 묶음으로 넘어가게 합니다. 그래서 한 주제에만 몰리지 않고 여러 주제의 청크를 고루 검사합니다.

<img src="images/03_lazy_budget.png" width="1050" alt="LazyGraphRAG의 질문 처리: 질문을 하위 질의 3개로 나누고, 청크 유사도로 커뮤니티 순위(1~4)를 매긴 뒤, LLM이 청크를 관련과 무관으로 나눕니다. 관련 청크로 답하며, 예산 막대(12, 24, 문서 전체)에서 예산 24를 다 쓰면 검사를 멈춥니다.">

전역 질문(자료 전체에 흩어진 내용을 모아야 하는 질문) G1 "조선 시대 선비들이 학문을 닦거나 유학을 가르치고 기리던 곳을 모두 꼽아 주세요"를 예산 24로 처리했을 때 판정 3회로 11편을 검사한 뒤, 관련 청크가 더 나오지 않아 예산을 다 쓰기 전에 멈췄습니다.

예산 24에서는 모든 질문이 예산을 다 쓰기 전에 멈췄습니다. 그래서 예산을 키워도 근거가 꼭 늘지는 않았습니다(관광 전역 질문의 근거 문서 포함률, 정답 근거 문서 가운데 넘긴 비율: 예산 12 10/14, 24 12/14, 48 12/14). 예산은 검사할 청크가 예산보다 훨씬 많은 큰 자료에서 근거 범위와 비용을 함께 정합니다.

### 1-3. 실무에서 쓰는 방법

- **공개 상태:** 방법론만 공개되었고 공개 구현은 없습니다. `graphrag` 3.1.0의 검색은 Local·Global·DRIFT·Basic뿐입니다.
- **실무 활용:** **질문 때 예산만큼 LLM으로 관련성을 검사한다**는 아이디어를 빌려, 벡터 검색 상위 문서를 LLM으로 한 번 더 걸러 근거만 넘깁니다.

**확인 퀴즈**: 배포 때마다 고쳐지는 운영 매뉴얼처럼 문서는 하루에도 여러 번 바뀌고 질문은 드물게 들어오는 사내 자료라면, Microsoft GraphRAG와 LazyGraphRAG 가운데 어느 쪽의 LLM 비용이 적을까요?

<details><summary>정답 보기</summary>

LazyGraphRAG입니다. 다시 색인해도 LLM 호출이 0회이고, LLM 비용은 드문 질문 때만 듭니다. 질문이 아주 많아지면 질문마다 관련성 검사 호출이 드는 LazyGraphRAG의 총비용이 더 커질 수 있습니다.

</details>

## 2. HippoRAG 2: 질문과 관련된 구에서 점수를 퍼뜨리는 연상 검색

HippoRAG는 질문과 관련된 구(개체 이름이나 값)에서 출발하는 PPR(Personalized PageRank: 시작 노드 가까이에 점수가 모이는 PageRank)로 문서 점수를 매깁니다. 이름은 단서 하나로 기억을 떠올리는 뇌의 해마(hippocampus)에서 왔습니다. 2024년 HippoRAG에 이어 2025년 [HippoRAG 2](https://arxiv.org/abs/2502.14802)가 나왔고, 이 부록은 최신판인 HippoRAG 2를 설명합니다.

### 2-1. 색인: 트리플에서 구 노드와 문서 노드 만들기

1. 문서마다 LLM으로 트리플(주어, 관계, 목적어)을 뽑습니다.
2. 주어와 목적어를 **구**(phrase) 노드로 두고, 관계로 잇습니다.
3. 문서도 노드로 두고, 그 문서의 트리플에 나온 구와 잇습니다.
4. 임베딩이 비슷한 구(코사인 유사도 0.8 이상)는 동의어 엣지로 잇습니다.

예를 들어 도서관 자료에 두 문서가 있다고 합시다(아래 그림과 같은 가상 예시).

- 문서 A 「신간 소개」: "『바다 일기』는 윤서진 작가가 쓴 소설이다." -> 트리플 `(바다 일기, 작가, 윤서진)`
- 문서 B 「도서관 행사」: "윤서진 작가가 10월에 시립도서관에서 강연한다." -> 트리플 `(윤서진, 강연 장소, 시립도서관)`

두 문서에 모두 나온 구 `윤서진`은 노드 하나입니다. 그래서 그래프는 이렇게 이어집니다.

- **구 노드끼리:** `바다 일기` -(작가)- `윤서진` -(강연 장소)- `시립도서관`
- **문서 노드와 구:** 문서 A는 `바다 일기`·`윤서진`과, 문서 B는 `윤서진`·`시립도서관`과 이어집니다.
- 결과적으로 문서 A와 문서 B가 `윤서진`을 거쳐 연결됩니다.

### 2-2. 질문: 트리플로 시드를 정하고 PPR로 문서 고르기

1. 질문 전체를 임베딩해 가장 비슷한 트리플 5개를 찾습니다.
2. LLM이 그 가운데 질문과 관련 없는 트리플을 걸러 냅니다. 남은 트리플이 없으면 그래프를 쓰지 않고 임베딩 검색 결과를 그대로 씁니다.
3. 남은 트리플의 구를 **시드**(PPR 시작 노드)로 둡니다. 문서 노드도 질문과의 임베딩 유사도에 작은 가중치(0.05)를 곱해 시드로 더합니다.
4. 시드에서 PPR을 돌려 점수를 이웃 구와 문서 노드로 퍼뜨립니다. `dampingFactor`(점수가 이웃으로 계속 퍼질 확률)는 0.5로, 점수가 시드 가까이에 모입니다.
5. 문서를 점수 순으로 골라 답변 모델에 넘깁니다. PPR에는 LLM을 쓰지 않습니다.

<img src="images/03_hipporag_ppr.png" width="1050" alt="HippoRAG 질문 처리: 질문 '바다 일기 작가의 강연은 어디서?'의 개체 '바다 일기'와 '강연'이 시드가 되고, 드문 개체일수록 큰 시드 가중치를 받습니다. 점수가 관계를 따라 작가 '윤서진'으로 퍼져, 질문과 닮지 않은 연결 구절이 구절 점수 순위 2위에 듭니다. dampingFactor 0.5는 점수를 시드 근처에 머물게 합니다.">

같은 예시로 질문 "『바다 일기』 작가의 강연은 어디서 하나요?"를 검색하면, 질문과 가장 비슷한 트리플 `(바다 일기, 작가, 윤서진)`이 남아 `바다 일기`와 `윤서진`이 시드가 됩니다. 점수가 `윤서진`에서 문서 B로 퍼져, 질문에 "윤서진"도 "시립도서관"도 없는데 문서 B가 상위에 듭니다. 벡터 검색은 질문 문장과 비슷한 문서 A는 찾지만, 질문과 표현이 다른 문서 B는 놓치기 쉽습니다. 이렇게 문서 두 편을 이어 읽어야 하는 멀티홉 질문에서 HippoRAG가 유리합니다.

### 2-3. 표기 통일이 중요한 이유

점수는 같은 구 노드를 거쳐야 다음 문서로 퍼집니다. 같은 고개가 한 문서에서는 `노고단 고개`, 다른 문서에서는 `노고단고개`로 적히면 다른 노드가 되어 점수가 끊깁니다. 사내 문서의 `point-wallet`과 `포인트 지갑 API`도 마찬가지입니다.

<img src="images/03_hipporag_naming.png" width="1050" alt="같은 작가가 한 문서에는 '윤서진', 다른 문서에는 '윤 작가'로 적히면 다른 노드라 점수가 퍼지지 않습니다. 동의어 엣지로 두 노드를 이으면 두 문서 모두 점수를 받습니다. HippoRAG 2는 비슷한 이름을 동의어 엣지로 이어 이 문제를 줄입니다.">

- **HippoRAG 2:** 임베딩이 비슷한 구(코사인 유사도 0.8 이상)를 동의어 엣지로 이어 표기 차이를 흡수합니다.
- **벡터 검색과 비교:** 표기를 통일한 관광 자료에서는 벡터 검색보다 근거를 더 넘겼고(10/11 대 7/11), 장애 지식베이스에서는 같았습니다(8/10 대 8/10).

### 2-4. 실무에서 쓰는 방법

- **공개 상태:** 저자의 연구 코드(알파)만 있어 운영 환경에 쓰기에는 이릅니다.
- **실무 활용:** 운영 중인 지식 그래프에서 질문과 관련된 노드를 시드로 PPR을 돌리는 아이디어를 빌립니다. Neo4j GDS의 `gds.pageRank.stream`에서 `sourceNodes`에 시드를 넘기면 됩니다.
- **전제 조건:** 같은 대상이 같은 구 노드로 이어져야 합니다(표기 통일이나 동의어 엣지).

**확인 퀴즈**: HippoRAG의 멀티홉 근거 문서 포함률이 벡터 검색보다 낮게 나왔다면, 이 방식을 후보에서 빼기 전에 무엇부터 확인해야 할까요?

<details><summary>정답 보기</summary>

어느 구 노드가 시드가 되었는지부터 확인합니다. 시드가 엉뚱한 노드에 붙었거나 같은 대상이 다른 이름의 노드로 나뉘어 있으면 점수가 다음 문서로 퍼지지 않습니다. 이때 원인은 PPR이 아니라 시드 연결과 표기에 있습니다.

</details>

## 3. 네 방식 비교: 측정 결과

교안 01·02와 같은 자료·골드 질문·모델(gpt-5.6-luna, text-embedding-3-large)로 미리 잰 결과입니다. HippoRAG는 표기를 통일한 트리플을 썼습니다. 넘긴 문서 수는 벡터·LightRAG가 5편, HippoRAG가 5편 이하(점수가 퍼진 문서만 넘김)이고, 나머지는 검색 결과에 따라 다릅니다.

### 3-1. LLM 비용이 드는 시점

가장 큰 차이는 **LLM을 색인 때 부르는지, 질문 때 부르는지**입니다.

<img src="images/03_cost_timing.png" width="1050" alt="네 방식이 LLM으로 하는 일을 색인 때와 질문 때로 나눈 그림입니다. Microsoft GraphRAG는 개체·관계 추출과 커뮤니티 보고서, LightRAG는 개체·관계 추출, HippoRAG는 트리플 추출을 색인 때 하고, LazyGraphRAG만 색인 때 LLM 없이 명사 그래프를 만듭니다. 질문 때는 모두 LLM으로 답변하며, LazyGraphRAG는 예산만큼 관련성 검사를 더합니다.">

**총비용 = 색인 비용 + 질문 1개당 비용 × 질문 수**

- 질문이 적고 문서가 자주 바뀌면 색인 비용이 크게 작용해, 색인에 LLM을 쓰지 않는 벡터 RAG나 LazyGraphRAG가 유리합니다.
- 질문이 많으면 질문 1개당 비용이 크게 작용해, 질문 때 LLM을 적게 부르는 방식이 유리합니다.

### 3-2. 색인과 문서 추가에 드는 LLM 호출 수

<img src="images/03_index_cost.png" width="1050" alt="본실습 문서 32편 색인과 문서 1편 추가에 든 LLM 호출 수를 방식별로 비교한 막대 차트입니다.">

- **색인:** GraphRAG 147회, LightRAG 34회입니다. GraphRAG는 커뮤니티 보고서까지 씁니다. HippoRAG는 트리플 추출에 32회(문서마다 1회)입니다.
- **문서 1편 추가:** GraphRAG 5회, LightRAG 1회, 벡터 RAG 0회입니다. GraphRAG는 새 커뮤니티마다 커뮤니티 보고서까지 써서 더 많습니다.

### 3-3. 질문 1개당 비용

<img src="images/03_query_cost.png" width="1050" alt="방식별 질문 1개당 평균 입력 토큰과 LLM 호출 횟수를 로그 눈금 막대로 비교한 차트입니다. 벡터 기준선과 HippoRAG가 가장 적고 GraphRAG Global이 가장 많습니다.">

- **Global:** 커뮤니티 보고서를 여러 덩어리로 나눠 덩어리마다 LLM으로 읽고 합쳐, 질문 비용이 가장 큽니다.
- **DRIFT·LazyGraphRAG:** 질문 하나에 LLM을 여러 번 부릅니다. DRIFT는 후속 질문마다, LazyGraphRAG는 관련성 판정마다 부릅니다.
- **HippoRAG:** LLM은 개체 추출과 답변 2회뿐이라 벡터 검색과 비용이 거의 같습니다.

### 3-4. 질문 유형별 근거 문서 포함률

포함률은 문서를 많이 넘길수록 저절로 오르므로 넘긴 문서 수와 함께 읽습니다. 원문 청크의 문서만 세므로, 개체·관계 설명이나 커뮤니티 보고서에 담긴 정보는 들어가지 않습니다.

**LazyGraphRAG가 가장 높은 이유:** 예산이 커서가 아닙니다(예산 24도 다 쓰기 전에 멈췄습니다). 다른 방식은 상위 5편으로 잘라 넘기지만, LazyGraphRAG는 LLM이 관련 있다고 판정한 청크를 모두 넘겨, 전역 질문에서는 질문당 평균 9편을 넘겼습니다. 커뮤니티 순서대로 여러 주제를 고루 검사한 것도 도움이 됐습니다. 대신 판정에 LLM 호출이 더 듭니다.

<img src="images/03_evidence_by_qtype.png" width="1050" alt="단일 사실·멀티홉·전역 질문별로 여섯 방식의 근거 문서 포함률을 비교한 막대 차트입니다. 괄호 안은 질문 1개당 넘긴 문서 수의 평균입니다.">

- **단일 사실 질문:** 방식 사이 차이가 작아 가장 싼 벡터 검색으로 충분합니다.
- **멀티홉 질문:** 관광 자료에서는 LazyGraphRAG(11/11), DRIFT(10/11), HippoRAG(10/11)가 벡터(7/11)보다 원문 근거를 많이 넘겼습니다. 장애 지식베이스에서는 DRIFT(9/10)만 벡터(8/10)보다 많았습니다. Local은 이어 읽을 다음 문서가 자주 빠졌고(관광 5/11, 장애 6/10), LightRAG는 두 자료 모두 벡터 검색과 같았습니다.
- **전역 질문:** 두 자료를 합쳐 LazyGraphRAG가 정답 근거 41편 가운데 39편을 넘겨 가장 많이 찾았습니다(벡터 17편, LightRAG 17편, HippoRAG 15편). 다만 질문당 평균 9편을 넘겨, 차이의 일부는 문서를 더 넘긴 결과입니다. 장애 지식베이스에서는 문서 20편을 모두 검사해 27/27를 얻었습니다.
- **GraphRAG Global:** 원문 대신 커뮤니티 보고서를 모두 읽으므로 포함률이 없습니다. 근거 범위는 자료 전체지만 질문 1개에 LLM을 7~15회 부릅니다.